# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ku-ro-wa/flyrank-ml-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [11]:
# Setup
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN {HF_TOKEN})")
rel = "hf://datasets/FlyRank/internship-warehouse"

# Check schema
con.sql(f"DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 0").show()

# Basic count
con.sql(f"""
    SELECT COUNT(*) AS n_rows
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").show()


┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┐
│  n_rows  │
│  int64   │
├──────────┤
│ 78835655 │
└──────────┘



## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signals:
- CTR (low(ctr<0.005), medium (0.005 ≤ ctr < 0.02), high (ctr ≥ 0.02))
- GSC data‑true (data present) / false (no GSC data)

Rule:
score = 0.6 if CTR<0.005 else 0, +0.4 if total_impr<200 else 0'
Reason codes - low_ctr_low_volume, low_ctr, low_volume, or normal
action_label - boost, optimize_ctr, increase_exposure, or none

In [15]:
# Signal checks
con.sql(f"""
    WITH recent AS (
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date >= (
        SELECT MAX(report_date) - INTERVAL '90 DAY'
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    )
), agg AS (
    SELECT content_hash_id,
           SUM(gsc_impressions) AS total_impr,
           SUM(gsc_clicks)    AS total_clicks,
           CASE WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_clicks)::DOUBLE / SUM(gsc_impressions)
                ELSE 0 END AS ctr
    FROM recent
    GROUP BY content_hash_id
)
SELECT
    CASE
        WHEN ctr < 0.005  THEN 'low'
        WHEN ctr < 0.02   THEN 'medium'
        ELSE                   'high'
    END AS ctr_bucket,
    COUNT(*) AS n
FROM agg
GROUP BY ctr_bucket;
""").show()

con.sql(f"""
    SELECT
      gsc_data_available,
      COUNT(*) AS n
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE gsc_impressions IS NOT NULL
    GROUP BY gsc_data_available;
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────┐
│ ctr_bucket │   n    │
│  varchar   │ int64  │
├────────────┼────────┤
│ high       │  11805 │
│ medium     │  31167 │
│ low        │ 366354 │
└────────────┴────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────┬──────────┐
│ gsc_data_available │    n     │
│      boolean       │  int64   │
├────────────────────┼──────────┤
│ false              │ 49767598 │
│ true               │ 28970051 │
└────────────────────┴──────────┘



Results:
- CTR ('low' bucket contained the majority)
- GSC data available flag (FALSE was the majority)

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [19]:
con.sql(f"""
    WITH recent AS (
    SELECT *
    FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
    WHERE report_date >= (SELECT MAX(report_date) - INTERVAL '90 DAY' FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet'))
),
agg AS (
    SELECT content_hash_id,
           SUM(gsc_impressions) AS total_impr,
           SUM(gsc_clicks)    AS total_clicks,
           CASE WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_clicks)::DOUBLE / SUM(gsc_impressions)
                ELSE 0 END AS ctr,
           AVG(gsc_avg_position) AS avg_position
    FROM recent
    GROUP BY content_hash_id
)
SELECT content_hash_id,
       total_impr,
       ctr,
       avg_position,
       (CASE WHEN ctr < 0.005 THEN 0.6 ELSE 0 END) +
       (CASE WHEN total_impr < 200 THEN 0.4 ELSE 0 END) AS score,
       CASE
           WHEN ctr < 0.005 AND total_impr < 200 THEN 'low_ctr_low_volume'
           WHEN ctr < 0.005                        THEN 'low_ctr'
           WHEN total_impr < 200                    THEN 'low_volume'
           ELSE 'normal'
       END AS reason_code,
       CASE
           WHEN ctr < 0.005 AND total_impr < 200 THEN 'boost'
           WHEN ctr < 0.005                        THEN 'optimize_ctr'
           WHEN total_impr < 200                    THEN 'increase_exposure'
           ELSE 'none'
       END AS action_label
FROM agg
ORDER BY score DESC, total_impr ASC
LIMIT 20;
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────────────┬────────────┬────────┬──────────────┬───────────────┬────────────────────┬──────────────┐
│     content_hash_id      │ total_impr │  ctr   │ avg_position │     score     │    reason_code     │ action_label │
│         varchar          │   int128   │ double │    double    │ decimal(12,1) │      varchar       │   varchar    │
├──────────────────────────┼────────────┼────────┼──────────────┼───────────────┼────────────────────┼──────────────┤
│ content_8c57bea9233eb9f7 │          0 │    0.0 │         NULL │           1.0 │ low_ctr_low_volume │ boost        │
│ content_e15de37a58b89469 │          0 │    0.0 │         NULL │           1.0 │ low_ctr_low_volume │ boost        │
│ content_a94a69b616a89fb8 │          0 │    0.0 │         NULL │           1.0 │ low_ctr_low_volume │ boost        │
│ content_82286860adc2be3a │          0 │    0.0 │         NULL │           1.0 │ low_ctr_low_volume │ boost        │
│ content_cd2e0de362d4d7de │          0 │    0.0 │      

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

So most of what is asked can already be seen in the output of the code cell above this one, though I do have a couple of notes. Mainly that since 'score' here acts as a warning indicator, aka it behaves opposite to the common view that a higher score == better performance.

That's why its not much surprise that the pages that rank the highest given this context are the ones performing the worst in that both their CTR and total impressions == 0 which is why they are flagged with having problems with both in their reason_code.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

###Where the current analysis could be incorrect or misleading:
- Zero‑impression rows (total_impr = 0): The CTR formula falls back to 0, which makes these rows look like extremely low‑CTR even though there is no data to judge performance.

- Missing GSC data: When the flag is false we still compute CTR and volume from the raw gsc_impressions column; many of those rows may contain only NULL or 0 values, inflating the “low‑volume/low‑CTR” pattern.

- Short‑term volatility: 	Aggregating exactly the last 90 days can be noisy for low‑traffic pages; a single day with a few clicks can swing the CTR dramatically.

- Bias toward “low‑volume": The rule gives a fixed 0.4 weight to any page with < 200 impressions, regardless of whether the low‑CTR condition is also present. This can over‑prioritize pages that simply lack traffic, even if their CTR is acceptable.

- Single‑metric focus: 	Only GSC‑impressions/clicks and the gsc_data_available flag are used. Other signals (e.g., GA4 sessions, AI‑traffic, scroll events) might contradict the CTR‑based view.


###Confirmation that no product flags or future windows leaked:
- No future dates
- Only allowed flag – the rule uses the single product flag gsc_data_available
- No future‑window or label‑derived inputs - All columns (gsc_impressions, gsc_clicks, gsc_avg_position) are observed today.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.